In [2]:
"""
MODELO 3/12 — Naive Bayes (NB)

Replica _IM_NaiveBayes.R:
    - Lê {Code}_DatasetNew.csv (features já normalizadas)
    - ANTES do WFA: filtra features com variância > 1e-4
      (model_data1 = model_data1[, var > 1e-4])
    - WFA: d1=250, d2=5, janela deslizante
    - klaR::NaiveBayes (Gaussian Naive Bayes)
    - predict retorna classes (1,2 factor levels) → converte via -1
    - Salva {Code}_TradeSignal_NB.csv

Uso:
    python 04_model_NB.py
"""

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ===================== CONFIGURAÇÃO =====================
BASE_DIR = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICSMI")
SEC_NAMES = BASE_DIR / ".NewB3_pruned.csv"

TRAIN_SIZE = 250
TEST_SIZE = 5
VAR_THRESHOLD = 1e-4  # filtro de variância do R
# ========================================================


def read_codes(path: Path) -> list:
    df = pd.read_csv(path, dtype=str, encoding="utf-8-sig")
    return df["Code"].str.strip().str.upper().tolist()


def run_wfa_nb(code: str, base_dir: Path) -> dict:
    """Executa Walk-Forward Analysis com Naive Bayes para um ticker."""
    infile = base_dir / f"{code}_DatasetNew_MI.csv"
    outfile = base_dir / f"{code}_TradeSignal_NB_MI.csv"

    if outfile.exists():
        return {"Code": code, "status": "skipped", "signals": 0}

    if not infile.exists():
        return {"Code": code, "status": "no_DatasetNew", "signals": 0}

    try:
        df = pd.read_csv(infile, encoding="utf-8-sig")
    except Exception as e:
        return {"Code": code, "status": f"read_error: {e}", "signals": 0}

    if df.shape[1] < 3:
        return {"Code": code, "status": "too_few_columns", "signals": 0}

    date_col = df.columns[0]
    label_col = df.columns[-1]

    # --- Filtro de variância ANTES do WFA (idêntico ao R) ---
    # R: model_data1 = model_data[,-c(1,n)]
    #    model_data1 = model_data1[, var > 1e-4]
    features_all = df.iloc[:, 1:-1]
    variances = features_all.var(axis=0, ddof=0)
    keep_cols = variances[variances > VAR_THRESHOLD].index.tolist()

    if not keep_cols:
        return {"Code": code, "status": "no_features_after_var_filter", "signals": 0}

    # Reconstruir dataframe filtrado (R: model_datax = data.frame(X, features_filtradas, Label))
    df_filtered = pd.concat([df[[date_col]], df[keep_cols], df[[label_col]]], axis=1)

    feature_cols = keep_cols

    # --- Alinhamento WFA (idêntico ao R) ---
    M = len(df_filtered)
    if M < TRAIN_SIZE + TEST_SIZE:
        return {"Code": code, "status": f"too_few_rows ({M})", "signals": 0}

    Q = (M - TRAIN_SIZE) // TEST_SIZE
    H = (M - TRAIN_SIZE) - TEST_SIZE * Q
    df_filtered = df_filtered.iloc[H:].reset_index(drop=True)

    dates = df_filtered[date_col].values
    X = df_filtered[feature_cols].values.astype(float)
    y = df_filtered[label_col].values.astype(int)

    predict_signal = []
    predict_dates = []

    # --- Loop WFA ---
    for i in range(Q):
        train_start = i * TEST_SIZE
        train_end = train_start + TRAIN_SIZE
        test_start = train_end
        test_end = test_start + TEST_SIZE

        X_train = X[train_start:train_end]
        y_train = y[train_start:train_end]
        X_test = X[test_start:test_end]
        test_dates = dates[test_start:test_end]

        if len(np.unique(y_train)) < 2:
            preds = [int(y_train[0])] * len(X_test)
        else:
            try:
                # klaR::NaiveBayes = Gaussian Naive Bayes
                clf = GaussianNB()
                clf.fit(X_train, y_train)
                preds = clf.predict(X_test).astype(int).tolist()
            except Exception:
                preds = [0] * len(X_test)

        predict_signal.extend(preds)
        predict_dates.extend(test_dates)

    # --- Salvar ---
    if predict_signal:
        df_out = pd.DataFrame({"Date": predict_dates, "pre_signal": predict_signal})
        df_out.to_csv(outfile, index=False, encoding="utf-8-sig")

    return {"Code": code, "status": "ok", "signals": len(predict_signal)}


def main():
    codes = read_codes(SEC_NAMES)
    print(f"Modelo: Naive Bayes (GaussianNB)")
    print(f"WFA: d1={TRAIN_SIZE}, d2={TEST_SIZE}")
    print(f"Filtro de variância: > {VAR_THRESHOLD}")
    print(f"Tickers: {len(codes)}\n")

    report = []
    for code in tqdm(codes, desc="NB Walk-Forward"):
        result = run_wfa_nb(code, BASE_DIR)
        report.append(result)

    report_df = pd.DataFrame(report)
    report_df.to_csv(BASE_DIR / "model_NB_report.csv", index=False, encoding="utf-8-sig")

    n_ok = (report_df["status"] == "ok").sum()
    n_skip = (report_df["status"] == "skipped").sum()
    avg_signals = report_df.loc[report_df["status"] == "ok", "signals"].mean()

    print(f"\n{'='*50}")
    print(f"Concluído: {n_ok} processados, {n_skip} já existiam.")
    print(f"Média de sinais por ação: {avg_signals:.0f}")
    print(f"Relatório: model_NB_report.csv")


if __name__ == "__main__":
    main()

Modelo: Naive Bayes (GaussianNB)
WFA: d1=250, d2=5
Filtro de variância: > 0.0001
Tickers: 239



NB Walk-Forward: 100%|██████████| 239/239 [00:33<00:00,  7.06it/s]


Concluído: 239 processados, 0 já existiam.
Média de sinais por ação: 564
Relatório: model_NB_report.csv
